In [26]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
现代汉语词典词性标注预处理脚本
输入：
    path_1: 《现代汉语词典》txt文件路径，每行一个词（纯词表，无词性）
    path_2: 《366万常用中文词汇整理》txt文件路径，格式：词语\t词性\t词频
输出：
    output.csv: 合并后的词条，包含词、词性、词频、来源标记
"""
import re
import os
import csv
import sys
import pandas as pd

def sample(dic, n=10):

    from itertools import islice

    return list(islice(dic.items(), n))

def pos_tag_bind(tag, standard_name="ctb"):
    """
    将词性标签映射到粗略分类。
    
    参数:
        tag (str): 原始词性标签
        standard_name (str): 标注标准，可选 "ctb" (北大标准) 或 "ict" (ICTCLAS标准)
    
    返回:
        str: 映射后的粗略词性，若无法映射则返回原标签
    """
    # 北大词性标注集 (CTB) 映射表
    ctb_map = {
        # 名词及其子类
        'n': 'n', 'nr': 'n', 'ns': 'n', 'nt': 'n', 'nz': 'n', 'Ng': 'n',
        # 动词及其子类
        'v': 'v', 'vi': 'v', 'vd': 'v', 'vn': 'v',
        # 形容词及其子类
        'a': 'a', 'ad': 'a', 'an': 'a',
        # 区别词
        'b': 'b',
        # 状态词
        'z': 'z',
        # 副词
        'd': 'd',
        # 代词
        'r': 'r',
        # 数词/量词
        'm': 'm', 'mq': 'mq', 'q': 'q',
        # 介词
        'p': 'p',
        # 连词
        'c': 'c',
        # 助词
        'u': 'u',
        # 方位词
        'f': 'f',
        # 叹词
        'e': 'e',
        # 拟声词
        'o': 'o',
        # 前/后缀成分
        'h': 'h', 'k': 'k',
        # 习用语
        'l': 'l',
        # 成语
        'i': 'i',
        # 略语
        'j': 'j',
        # 时间词
        't': 't', 'tg': 't',
        # 处所词
        's': 's',
        # 字符串
        'x': 'x',
        # 标点符号
        'w': 'w',
    }
    
    # ICTCLAS 词性标注集 (ICT) 映射表
    ict_map = {
        # 名词及其子类 (ICTCLAS 有更细的划分)
        'n': 'n', 'nr': 'n', 'ns': 'n', 'nt': 'n', 'nz': 'n', 'ng': 'n',
        'nf': 'n', 'ni': 'n', 'nl': 'n', 'nh': 'n', 'nhd': 'n', 'nhm': 'n',
        'nnd': 'n', 'nnt': 'n', 'nn': 'n', 'nba': 'n', 'nbc': 'n', 'nbf': 'n',
        'nbi': 'n', 'nbs': 'n', 'nbt': 'n', 'nca': 'n', 'ncb': 'n', 'ncc': 'n',
        'ncd': 'n', 'ncf': 'n', 'ncj': 'n', 'nck': 'n', 'ncl': 'n', 'ncm': 'n',
        'ncn': 'n', 'nco': 'n', 'ncp': 'n', 'ncq': 'n', 'ncr': 'n', 'ncs': 'n',
        'nct': 'n', 'ncu': 'n', 'ncv': 'n', 'ncw': 'n', 'ncx': 'n', 'ncy': 'n',
        'ncz': 'n', 'nd': 'n', 'ne': 'n', 'ng': 'n', 'nhc': 'n', 'nhk': 'n',
        'nhm': 'n', 'nho': 'n', 'nhp': 'n', 'nhs': 'n', 'nht': 'n', 'nhy': 'n',
        'nif': 'n', 'nis': 'n', 'nit': 'n', 'niy': 'n', 'njy': 'n', 'nk': 'n',
        'nm': 'n', 'nmc': 'n', 'nmf': 'n', 'nmh': 'n', 'nmm': 'n', 'nmo': 'n',
        'nms': 'n', 'nmz': 'n', 'nn': 'n', 'nna': 'n', 'nnb': 'n', 'nnc': 'n',
        'nnd': 'n', 'nne': 'n', 'nnf': 'n', 'nng': 'n', 'nnh': 'n', 'nni': 'n',
        'nnj': 'n', 'nnk': 'n', 'nnl': 'n', 'nnm': 'n', 'nnn': 'n', 'nno': 'n',
        'nnp': 'n', 'nnq': 'n', 'nnr': 'n', 'nns': 'n', 'nnt': 'n', 'nnu': 'n',
        'nnv': 'n', 'nnw': 'n', 'nnx': 'n', 'nny': 'n', 'nnz': 'n', 'np': 'n',
        'nq': 'n', 'nr': 'n', 'nr1': 'n', 'nr2': 'n', 'nrf': 'n', 'nrg': 'n',
        'nrj': 'n', 'ns': 'n', 'nsf': 'n', 'nst': 'n', 'nt': 'n', 'ntc': 'n',
        'ntcb': 'n', 'ntcf': 'n', 'ntch': 'n', 'ntd': 'n', 'nte': 'n', 'ntf': 'n',
        'nth': 'n', 'ntj': 'n', 'ntl': 'n', 'nto': 'n', 'ntp': 'n', 'nts': 'n',
        'ntt': 'n', 'ntu': 'n', 'ntw': 'n', 'ntz': 'n', 'nw': 'n', 'nx': 'n',
        'ny': 'n', 'nz': 'n',
        # 动词及其子类
        'v': 'v', 'vi': 'v', 'vd': 'v', 'vn': 'v', 'vq': 'v', 'vf': 'v',
        'vl': 'v', 'vp': 'v',
        # 形容词及其子类
        'a': 'a', 'ad': 'a', 'an': 'a', 'al': 'a',
        # 区别词
        'b': 'b', 'bl': 'b',
        # 状态词
        'z': 'z',
        # 副词
        'd': 'd', 'dl': 'd',
        # 代词
        'r': 'r', 'rr': 'r', 'ry': 'r', 'ryv': 'r', 'rz': 'r', 'rzt': 'r',
        'rzs': 'r', 'rzv': 'r',
        # 数词/量词
        'm': 'm', 'mq': 'mq', 'q': 'q', 'qv': 'q',
        # 介词
        'p': 'p',
        # 连词
        'c': 'c', 'cc': 'c',
        # 助词
        'u': 'u', 'udh': 'u', 'uls': 'u',
        # 方位词
        'f': 'f',
        # 叹词
        'e': 'e',
        # 拟声词
        'o': 'o',
        # 前/后缀成分
        'h': 'h', 'k': 'k',
        # 习用语
        'l': 'l',
        # 成语
        'i': 'i',
        # 略语
        'j': 'j',
        # 时间词
        't': 't', 'tg': 't',
        # 处所词
        's': 's',
        # 字符串
        'x': 'x',
        # 标点符号
        'w': 'w',
        # 特殊标注
        'comb': 'comb',   # 组合词（保留）
        'nw': 'nw',       # 无法确定词性（保留）
    }
    
    # 根据 standard_name 选择映射表
    if standard_name.lower() == "ctb":
        mapping = ctb_map
    elif standard_name.lower() == "ict":
        mapping = ict_map
    else:
        raise ValueError("standard_name 必须是 'ctb' 或 'ict'")
    
    # 返回映射后的标签，若未找到则返回原标签
    return mapping.get(tag, tag)

def load_366m_dict(path_2):
    """
    加载366万词库，构建 word -> (pos, freq) 的映射
    """
    word_map = {}
    try:
        with open(path_2, 'r', encoding='utf-8') as f:
            for line_num, line in enumerate(f, 1):
                line = line.strip()
                if not line:
                    continue
                parts = line.split('\t')
                if len(parts) != 3:
                    # 格式不符合预期，跳过并打印警告
                    print(f"警告：第{line_num}行格式错误，跳过: {line[:50]}...")
                    continue
                word, pos, freq_str = parts
                try:
                    freq = int(freq_str)
                except ValueError:
                    freq = 0
                # 如果同一个词出现多次（理论上不应出现），保留第一次出现的记录
                if word not in word_map:
                    word_map[word] = (pos, freq)
    except FileNotFoundError:
        print(f"错误：找不到文件 {path_2}")
        sys.exit(1)
    except Exception as e:
        print(f"读取{path_2}时出错: {e}")
        sys.exit(1)
    print(f"成功加载366万词库，共 {len(word_map)} 个词条。")
    return word_map

def load_modern_dict(path_1):
    """
    加载《现代汉语词典》词表，返回词条列表（保持顺序）
    """
    words = []
    try:
        with open(path_1, 'r', encoding='utf-8') as f:
            for line in f:
                word = line.strip()
                if word:  # 忽略空行
                    words.append(word)
    except FileNotFoundError:
        print(f"错误：找不到文件 {path_1}")
        sys.exit(1)
    except Exception as e:
        print(f"读取{path_1}时出错: {e}")
        sys.exit(1)
    print(f"成功加载《现代汉语词典》词条，共 {len(words)} 个词。")
    return words

def annotate_modern_dict(words, word_map):
    """
    对现代汉语词典中的每个词进行词性标注
    返回列表，每个元素为 (word, pos, freq, matched_flag)
    """
    annotated = []
    matched_count = 0
    for word in words:
        if word in word_map:
            pos, freq = word_map[word]
            annotated.append((word, pos, freq, True))
            matched_count += 1
        else:
            # 未匹配：默认词性设为 nw（无法确定），词频为0
            annotated.append((word, 'nw', 0, False))
    print(f"匹配成功：{matched_count} / {len(words)} ({matched_count/len(words)*100:.2f}%)")
    return annotated

def save_to_csv(annotated, output_path):
    """
    将标注结果保存为CSV文件
    """
    try:
        with open(output_path, 'w', encoding='utf-8', newline='') as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow(['word', 'pos', 'freq', 'matched'])
            for word, pos, freq, matched in annotated:
                writer.writerow([word, pos, freq, 1 if matched else 0])
        print(f"结果已保存至: {output_path}")
    except Exception as e:
        print(f"保存CSV文件时出错: {e}")
        sys.exit(1)


In [2]:
word_map = load_366m_dict("D:/Documents/文化/汉语词典/360万中文词库/词典360万单词量.txt")

成功加载366万词库，共 3337673 个词条。


In [3]:
words = load_modern_dict("D:/Documents/文化/汉语词典/现代汉语词典（第7版）.txt")

成功加载《现代汉语词典》词条，共 58852 个词。


In [4]:
ann = annotate_modern_dict(words, word_map)

pattern = re.compile(r'[a-zA-Z0-9]')

ann2 = []

for word, pos, freq, r in ann:
    if r and len(word) == 2 and not pattern.search(word):
        ann2.append((word, pos, freq))

len(ann2)

匹配成功：57316 / 58852 (97.39%)


44008

In [21]:
# 转换为DataFrame并按freq降序排序
df = pd.DataFrame(ann2, columns=['word', 'pos', 'freq'])
df_sorted = df.sort_values('freq', ascending=False).reset_index(drop=True)

In [5]:
with open("D:/Documents/文化/汉语词典/SUBTLEX-CH-CHR", "r") as f:
    content = f.readlines()

chars = [line.rstrip("\n").split("\t") for line in content[2:]]
len(chars)

5937

In [6]:
with open("D:/Documents/文化/汉语词典/SUBTLEX-CH-WF_PoS", "r") as f:
    content = f.readlines()

wordpos = []
for line in content[1:]:
    linex = line.rstrip("\n").lstrip("\t")
    if linex:
        wordpos.append(linex.split("\t"))
len(wordpos)

207083

In [7]:
word_pos = {}
key = None

for x in wordpos[1:]:
    if x[0] == '@' and key and x[2] == key and key in word_pos:
        word_pos[key]['sens'].append((x[3], int(x[4])))
    else:
        key = x[0]
        word_pos[key] = {"freq": int(x[1]), 'sens': []}

In [8]:
tags2 = set([pos for k, (pos, n) in word_map.items()])

tags = []

for x in wordpos[1:]:
    if x[0] == '@' and x[3] not in tags:
        tags.append(x[3])

In [9]:
words_gen = list(word_pos.keys())

word1_gen = [word for word in words_gen if len(word) == 1]

[(w, word_pos[w]) for w in word1_gen[3600:3640]]

[('耀', {'freq': 29, 'sens': [('g', 29)]}),
 ('老', {'freq': 16451, 'sens': [('a', 15745), ('d', 687), ('g', 19)]}),
 ('考', {'freq': 815, 'sens': [('v', 814), ('g', 1)]}),
 ('者',
  {'freq': 11387,
   'sens': [('k', 10619),
    ('r', 478),
    ('y', 279),
    ('n', 7),
    ('g', 2),
    ('u', 2)]}),
 ('耆', {'freq': 5, 'sens': [('g', 5)]}),
 ('而', {'freq': 39381, 'sens': [('cc', 39380), ('u', 1)]}),
 ('耍', {'freq': 1824, 'sens': [('v', 1824)]}),
 ('耐', {'freq': 223, 'sens': [('v', 223)]}),
 ('耕', {'freq': 32, 'sens': [('v', 32)]}),
 ('耗', {'freq': 184, 'sens': [('v', 181), ('g', 3)]}),
 ('耙', {'freq': 13, 'sens': [('v', 10), ('n', 3)]}),
 ('耦', {'freq': 2, 'sens': [('n', 2)]}),
 ('耨', {'freq': 1, 'sens': [('n', 1)]}),
 ('耳', {'freq': 631, 'sens': [('n', 503), ('y', 128)]}),
 ('耶', {'freq': 4120, 'sens': [('y', 3912), ('b', 208)]}),
 ('耸', {'freq': 38, 'sens': [('g', 38)]}),
 ('耻', {'freq': 75, 'sens': [('g', 75)]}),
 ('耽', {'freq': 31, 'sens': [('nr', 31)]}),
 ('耿', {'freq': 3, 'sens': [('

In [18]:
with open("D:/Documents/文化/汉语词典/XDHYCD7th.txt", 'r', encoding='utf-8') as f:
    content = f.readlines()

In [20]:
# print("".join(content[13:60]))

tags1 = []
for line in content[13:]:
    for t in re.findall(r'〈(.*?)〉', line):
        if "〈" in t:
            print(line)
            continue
        if t not in tags1:
            tags1.append(t)

print(tags1)

['名', '方', '叹', '拟声', '形', '动', '助', '注意', '书', '量', '介', '口', '副', '代', '数', '古', '连']


In [37]:
import json 

with open("D:/Documents/文化/汉语词典/word.json","r",encoding="utf-8") as f:
    ziji = json.load(f)

In [ ]:
tags3 = []

for zi in ziji:
    for t in re.findall(r'〈(.*?)〉', zi['explanation'] + " " + zi['more']):
        if t not in tags3:
            tags3.append(t)

tags3
    

In [207]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import re
import json
import csv
from bs4 import BeautifulSoup

def clean_html(html_content):
    """去除HTML标签，提取纯文本，并清理多余空白"""
    soup = BeautifulSoup(html_content, 'html.parser')
    text = soup.get_text(separator='\n')
    # 去除多余空行和前后空白
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return '\n'.join(lines)

def process_tsv(input_file):
    results = []
    with open(input_file, 'r', encoding='utf-8') as f:
        for line in f.readlines():
            row = line.rstrip("\n").split("\t")
            if len(row) < 2:
                continue
            word = row[0].strip()
            html = "\t".join(row[1:])
            clean_text = clean_html(html)
            # clean_text = html
            results.append({
                'word': word,
                'text': clean_text
            })
    
    print(f"处理完成，共 {len(results)} 个词条")
    return results

results = process_tsv("D:/Documents/文化/汉语词典/新华字典.tsv")
xhzd = {}

for x in results:
    if x['word'] not in xhzd:
        xhzd[x['word']] = x['text']
    else:
        xhzd[x['word']] += "\n" + x['text']

xinhua = {}
for k, v in xhzd.items():
    if '详细解析' in v:
        xinhua[k] = v

处理完成，共 20915 个词条


In [22]:
print(xinhua['窕'])

基本解释
Basic explanation
窕　　tiǎo   ㄊㄧㄠˇ
1. 细：“小者不～。”
2. 有空隙：“充盈大宇而不～”。
3. 美好：“秦晋之间，凡美色，或谓之好，或谓之～”。
窕　　yáo   ㄧㄠˊ
妖艳，轻挑：～冶。
详细解析
Detailed explanation
◎ 窕 tiǎo
〈形〉
(1) (形声。从穴,兆声。本义:深邃)
(2) 同本义
[profound]
窕,深肆也。--《说文》
充盈大宇而不窕。--《荀子·赋》
入小而不偪,处大而不窕。--《淮南子》
(3) 又如:窕邃(幽深的样子);窕窕(幽深的样子)
(4) 未充满;间隙
[unreplenished]
七者布诸天下而不窕,内诸寻常之室而不塞。--《大戴礼记》
(5) 细;小
[small]
夫天子省风以作乐, 小者不窕。--《汉书》
(6) 虚浮不实
[showy and superficial]
语言辨,听之说(悦),不度于义,谓之窕言。--《韩非子》
(7) 美;美色
[beautiful]
不至于窕冶。--《荀子·礼论》
窕美也。美状为窕。--《方言二》
(8) 又如:窕窈(窈窕;美貌);窕儇(美貌,轻佻)
(9) 淫;过分
[excessive]。
(10) 如:窕名(虚名);窕言(虚假不实之言);窕货(来路不正的货物)
(11) 过剩的,多余的
[superfluous]
充盈大宇而不窕,入郄穴而不逼者与?--《荀子》
(12) 优美、雅致和高贵的
[gentle and graceful]。
(13) 如:窈窕
中文輸入法 Input Methods
仓颉 Cāngjié
JCLMO
郑码 Zhèngmǎ
WOVR
四角号码 Four Corner
30113
Unicode
U+7A95


In [120]:
pinyin_chars = (
    # 基本小写字母
    "abcdefghijklmnopqrstuvwxyz"
    # 带声调的小写元音
    "āáǎàōóǒòēéěèīíǐìūúǔù"
    # 小写 ü 及其声调变体
    "üǖǘǚǜ"
    # 小写 ê 及带声调的 m/n
    "ê ń ň ǹ"
    # 基本大写字母
    "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    # 带声调的大写元音
    "ĀÁǍÀŌÓǑÒĒÉĚÈĪÍǏÌŪÚǓÙ"
    # 大写 Ü 及其声调变体
    "ÜǕǗǙǛ"
    # 大写 Ê 及带声调的 M/N
    "Ê Ń Ň Ǹ"
)
pinyin_chars = ''.join(pinyin_chars.split())

In [121]:
pinyin_chars

'abcdefghijklmnopqrstuvwxyzāáǎàōóǒòēéěèīíǐìūúǔùüǖǘǚǜêńňǹABCDEFGHIJKLMNOPQRSTUVWXYZĀÁǍÀŌÓǑÒĒÉĚÈĪÍǏÌŪÚǓÙÜǕǗǙǛÊŃŇǸ'

In [144]:
def proc_zi(zi, x, fail=[]):
    # x = xinhua[zi]
    r = {}
    if '详细解析' in x:
        group = x.split("详细解析")
        bex = group[0].split("Basic explanation")[1].lstrip().lstrip("\n").lstrip()
        # try:
        dex = group[1].split("中文輸入法")[0].split("Detailed explanation")[1].lstrip().lstrip("\n").lstrip().split("◎")
        r['详解'] = []
        for ex in dex:
            if ex:
                box = {}
                box['义'] = []
                text = ex.strip()
                pinyin_ok = False
                for block in text[:12].split():
                    if all([s in pinyin_chars for s in block]):  # 纯拼音块存在
                        pinyin_ok = True
                if pinyin_ok:
                    group = re.split(r'([' + pinyin_chars + r']+)', text)
                    box['音'] = group[1].rstrip("\n").rstrip()
                    if "\n" in group[0]:
                        box["通"] = group[0].split("\n")[-1].strip().split("、")
                    text = "".join(group[2:])
                else:
                    box['音'] = '--'
                    for i in range(12):
                        if text[i] == "\n":
                            text = text[i:]
                            break
                if re.search(r'〈([^〉]+)〉', text) and re.search(r'〈([^〉]+)〉', text).group(0):
                    group = re.split(r'〈([^〉]+)〉', text)
                    box["类"] = group[1]
                    text = "".join(group[2:])
                if re.search(r'\((\d+)\)', text) and re.search(r'\((\d+)\)', text).group(0):
                    group = [i.strip().strip("\n").strip() for i in re.split(r'\(\d+\)', text)]
                    for item in group:
                        if item:
                            box["义"].append(item.split("\n"))
                else:
                    box["义"].append(text)
                r['详解'].append(box)
        # except Exception as e:
        #     # print(f"汉字：{zi}的详细解析内容无法提取。")
        #     r['详解'] = x.split("详细解析")[1:]
        #     fail.append(zi)
    else:
        bex = x.split("Basic explanation")[1].split("中文輸入法")[0].split("Detailed explanation")[1].lstrip().lstrip("\n").lstrip()

    r['基本'] = []
    box = None
    # try:
    if "\u3000\u3000" in bex:
        bex = re.sub('\u3000\u3000', '\n', bex)
    for line in bex.split("\n"):
        if not line.strip().strip(zi):
            if box is not None and '拼音' in box and box['拼音']:
                r['基本'].append(box)
            box = {}
            box['义'] = []
        elif re.match(r'[' + pinyin_chars + r']+\s+', line):
            group = line.split()
            pyline = True
            for s in group[0]:
                if s not in pinyin_chars:
                    pyline = False
                    box['义'].append(line.split("："))  # 不认为是拼音、注音内容，假定为义项内容
                    break
            if pyline:
                box['拼音'] = group[0]
                if len(group) > 1:
                    box['注音'] = group[1]
        elif line.strip().strip("\n"):
            if re.match(r'\d+?\.', line):
                text = line.split(".")[-1].lstrip()
            else:
                text = line
            box['义'].append(text.split("："))
    if box is not None and '拼音' in box and box['拼音']:
        r['基本'].append(box)
    # except Exception as e:
    #     # print(f"汉字：{zi}的基本解释内容无法提取。")
    #     r['基本'] = bex
    #     if zi not in fail:
    #         fail.append(zi)
    return r

r = proc_zi('呣', xinhua['呣'])
r

{'详解': [{'义': [['表示强烈感情、招呼、答应或疑问',
     '[um]',
     '“呣!…”这声音从他的心底冲了出来,但立刻被他的喉咙梗住了… --鲁彦《桥上》'],
    ['又如:呣,什么?'],
    ['另见 ?', '词性变化']],
   '音': '--'},
  {'义': [['表示应诺', '[um-hum]。如:呣,我知道了;呣,我就来'], ['另见 ?']], '音': '--'}],
 '基本': [{'义': [['◎ 古同“谋”，思虑。']], '拼音': 'móu', '注音': 'ㄇㄡˊ'},
  {'义': [['◎ 叹词。']], '拼音': 'm', '注音': 'ㄇˊ'},
  {'义': [['◎ 应答声', '～，我知道了。']], '拼音': 'm', '注音': 'ㄇˋ'}]}

In [220]:
# zi = '着'
fail = []
zige = {}
n = 12
for zi in xinhua:
    x = xinhua[zi]
    r = {}
    if '详细解析' in x:
        group = x.split("详细解析")
        bex = group[0].split("Basic explanation")[1].lstrip().lstrip("\n").lstrip()
        try:
            dex = group[1].split("中文輸入法")[0].split("Detailed explanation")[1].split("词性变化")[0].lstrip().lstrip("\n").lstrip().split("◎")
            r['详解'] = []
            for ex in dex:
                if ex:
                    box = {}
                    box['义'] = []
                    text = ex.strip()
                    pinyin_ok = False
                    for block in text[:n].split():
                        if all([s in pinyin_chars for s in block]):  # 纯拼音块存在
                            pinyin_ok = True
                    if pinyin_ok:
                        group = re.split(r'([' + pinyin_chars + r']+)', text)
                        box['音'] = group[1].rstrip("\n").rstrip()
                        if "\n" in group[0]:
                            tongjia = group[0].split("\n")[-1].strip()
                            if len(tongjia):
                                box["通"] = [t for t in tongjia.split("、") if t]
                        text = "".join(group[2:])
                    else:
                        box['音'] = '--'
                        for i in range(n):
                            if text[i] == "\n":
                                text = text[i:]
                                break
                    if re.search(r'〈([^〉]+)〉', text) and re.search(r'〈([^〉]+)〉', text).group(0):
                        group = re.split(r'〈([^〉]+)〉', text)
                        box["类"] = group[1]
                        text = "".join(group[2:])
                    if re.search(r'\((\d+)\)', text) and re.search(r'\((\d+)\)', text).group(0):
                        group = [i.strip().strip("\n").strip() for i in re.split(r'\(\d+\)', text)]
                        for item in group:
                            if item:
                                box["义"].append(item.split("\n"))
                    else:
                        box["义"].append(text)
                    r['详解'].append(box)
        except Exception as e:
            # print(f"汉字：{zi}的详细解析内容无法提取。")
            r['详解'] = x.split("详细解析")[1:]
            fail.append(zi)
    else:
        bex = x.split("Basic explanation")[1].split("中文輸入法")[0].split("Detailed explanation")[1].lstrip().lstrip("\n").lstrip()

    r['基本'] = []
    box = None
    try:
        if "\u3000\u3000" in bex:
            bex = re.sub('\u3000\u3000', '\n', bex)
        for line in bex.split("\n"):
            if not line.strip().strip(zi):
                if box is not None and '拼音' in box and box['拼音']:
                    r['基本'].append(box)
                box = {}
                box['义'] = []
            elif re.match(r'[' + pinyin_chars + r']+\s+', line):
                group = line.split()
                pyline = True
                for s in group[0]:
                    if s not in pinyin_chars:
                        pyline = False
                        box['义'].append(line.split("："))  # 不认为是拼音、注音内容，假定为义项内容
                        break
                if pyline:
                    box['拼音'] = group[0]
                    if len(group) > 1:
                        box['注音'] = group[1]
            elif line.strip().strip("\n"):
                if re.match(r'\d+?\.', line):
                    text = line.split(".")[-1].lstrip()
                else:
                    text = line
                box['义'].append(text.split("："))
        if box is not None and '拼音' in box and box['拼音']:
            r['基本'].append(box)
    except Exception as e:
        # print(f"汉字：{zi}的基本解释内容无法提取。")
        r['基本'] = bex
        if zi not in fail:
            fail.append(zi)

    zige[zi] = r
# box

In [221]:
zid = {}
for zi in zige:
    if zi in fail:
        continue
    zid[zi] = zige[zi]

len(zid)

8858

In [228]:
from outils import dump_cn_json_compact

dump_cn_json_compact("新华字典.json", zid)

In [226]:
import re
from typing import Dict, List, Any, Tuple

def score_sanity(zi: str, zidian: Dict[str, Any]) -> Dict[str, Any]:
    """
    检查单个词条，返回分数和问题列表。
    返回: {
        "score": int,           # 总扣分（分数越高问题越严重）
        "critical": List[str],
        "warning": List[str],
        "info": List[str]
    }
    """
    entry = zidian.get(zi)
    if not entry:
        return {
            "score": 100,   # 词条不存在，当作一个严重错误
            "critical": [f"词条「{zi}」在字典中不存在"],
            "warning": [],
            "info": []
        }

    result = {
        "score": 0,
        "critical": [],
        "warning": [],
        "info": []
    }

    def is_empty_str(s):
        return not isinstance(s, str) or len(s.strip()) == 0

    # ---------- 1. 基本解释检查 ----------
    basic_list = entry.get("基本")
    if not isinstance(basic_list, list):
        result["critical"].append(f"「基本」字段不是列表，类型为 {type(basic_list)}")
    else:
        if len(basic_list) == 0:
            result["warning"].append(f"「基本」列表为空（可能原始字典无基本解释）")
        for idx, basic in enumerate(basic_list):
            prefix = f"基本解释第{idx+1}项"
            if not isinstance(basic, dict):
                result["critical"].append(f"{prefix}：不是字典，类型为 {type(basic)}")
                continue

            pinyin = basic.get("拼音")
            if is_empty_str(pinyin):
                result["critical"].append(f"{prefix}：拼音缺失或为空")
            else:
                # 拼音格式检查（非空即可，不强制拼音合法性）
                pass

            senses = basic.get("义")
            if not isinstance(senses, list):
                result["critical"].append(f"{prefix}：「义」字段不是列表")
            elif len(senses) == 0:
                result["critical"].append(f"{prefix}：义项列表为空")
            else:
                for si, sense in enumerate(senses):
                    if len(sense) == 0:
                        result["critical"].append(f"{prefix} 第{si+1}个义项为空")
                        continue
                    if isinstance(sense, str):
                        definition = sense
                    elif isinstance(sense, list):
                        definition = sense[0]
                    else:
                        result["critical"].append(f"{prefix} 第{si+1}个义项：内容既不是列表也不是字符串")
                        continue
                    if is_empty_str(definition):
                        result["critical"].append(f"{prefix} 第{si+1}个义项：释义为空")
                    # 检查英文标记中的括号问题（info）
                    if len(sense) >= 2 and isinstance(sense[1], str):
                        eng = sense[1]
                        if "[" in eng and "]" in eng:
                            match = re.search(r'\[([^\]]*)\]', eng)
                            if match and len(match.group(1).strip()) == 0:
                                result["info"].append(f"{prefix} 第{si+1}个义项：英文标记为空括号")

            zhuyin = basic.get("注音")
            if zhuyin is not None and not isinstance(zhuyin, str):
                result["warning"].append(f"{prefix}：注音字段类型异常（应为字符串）")

    # ---------- 2. 详细解析检查 ----------
    detail_list = entry.get("详解")
    if not isinstance(detail_list, list):
        result["critical"].append(f"「详解」字段不是列表，类型为 {type(detail_list)}")
    else:
        if len(detail_list) == 0:
            result["warning"].append(f"「详解」列表为空（可能原始字典无详细解析）")
        for idx, detail in enumerate(detail_list):
            prefix = f"详解第{idx+1}项"
            if not isinstance(detail, dict):
                result["critical"].append(f"{prefix}：不是字典，类型为 {type(detail)}")
                continue

            pinyin = detail.get("音")
            if is_empty_str(pinyin):
                result["critical"].append(f"{prefix}：拼音缺失或为空")

            tong = detail.get("通")
            if tong is not None:
                if not isinstance(tong, list):
                    result["critical"].append(f"{prefix}：「通」字段存在但不是列表")
                elif len(tong) == 0:
                    result["critical"].append(f"{prefix}：「通」列表为空")
                else:
                    for t in tong:
                        if is_empty_str(t):
                            result["critical"].append(f"{prefix}：「通」列表中含有空字符串")

            pos = detail.get("类")
            if pos is not None and is_empty_str(pos):
                result["critical"].append(f"{prefix}：「类」字段存在但为空字符串")

            senses = detail.get("义")
            if not isinstance(senses, list):
                result["critical"].append(f"{prefix}：「义」字段不是列表")
            elif len(senses) == 0:
                result["critical"].append(f"{prefix}：义项列表为空")
            else:
                for si, sense in enumerate(senses):
                    if len(sense) == 0:
                        result["warning"].append(f"{prefix} 第{si+1}个义项：空列表")
                        continue
                    if isinstance(sense, list):
                        first = sense[0]
                    elif isinstance(sense, str):
                        first = sense
                    else:
                        result["critical"].append(f"{prefix} 第{si+1}个义项：内容不是列表也不是字符串")
                        continue
                    if isinstance(first, str) and first.strip() == "词性变化":
                        result["critical"].append(f"{prefix} 第{si+1}个义项：误将「词性变化」作为义项内容")
                    if is_empty_str(first):
                        result["critical"].append(f"{prefix} 第{si+1}个义项：释义为空")
                    if isinstance(first, str) and "另见" in first:
                        if not re.search(r'[a-zāáǎàēéěèīíǐìōóǒòūúǔùǖǘǚǜ]', first):
                            result["warning"].append(f"{prefix} 第{si+1}个义项：「另见」标记但未发现拼音目标")
                    if isinstance(first, str) and first.startswith("又如:"):
                        for elem in sense[1:]:
                            if isinstance(elem, str) and elem.startswith("[") and "]" in elem:
                                result["info"].append(f"{prefix} 第{si+1}个义项：「又如」义项中不应包含英文标记，可能解析错误")
                                break
                    # 英文标记格式小问题（info）
                    if isinstance(sense, list) and len(sense) >= 2 and isinstance(sense[1], str):
                        eng = sense[1]
                        if "[" in eng and "]" in eng:
                            match = re.search(r'\[([^\]]*)\]', eng)
                            if match and len(match.group(1).strip()) == 0:
                                result["info"].append(f"{prefix} 第{si+1}个义项：英文标记为空括号")
                        # 检查是否有孤立的方括号（如 "[put in" 缺少右括号）- 视为 warning
                        if eng.count('[') != eng.count(']'):
                            result["warning"].append(f"{prefix} 第{si+1}个义项：英文标记方括号不匹配")

    # ---------- 3. 跨部分一致性 ----------
    basic_pinyins = {b.get("拼音").lower() for b in basic_list if isinstance(b, dict) and b.get("拼音")}
    detail_pinyins = {d.get("音").lower() for d in detail_list if isinstance(d, dict) and d.get("音")}
    if basic_pinyins and detail_pinyins:
        if not basic_pinyins.intersection(detail_pinyins):
            if "--" in basic_pinyins or "--" in detail_pinyins:
                result["warning"].append(f"基本解释与详细解析的拼音集合完全没有交集（基本：{basic_pinyins}，详细：{detail_pinyins}）")
            else:
                sane = True
                for py in basic_pinyins:
                    if not any([i in pinyin_chars for i in py]):
                        sane = False
                        break
                for py in detail_pinyins:
                    if not any([i in pinyin_chars for i in py]):
                        sane = False
                        break
                if sane:
                    result["warning"].append(f"基本解释与详细解析的拼音集合完全没有交集（基本：{basic_pinyins}，详细：{detail_pinyins}）")
                else:
                    result["critical"].append(f"基本解释与详细解析的拼音集合完全没有交集（基本：{basic_pinyins}，详细：{detail_pinyins}）")
    elif basic_pinyins and not detail_pinyins:
        result["warning"].append("有基本解释拼音，但详细解析为空或无效")
    elif not basic_pinyins and detail_pinyins:
        result["warning"].append("有详细解析，但基本解释为空或无效")

    # ---------- 4. 计算总分 ----------
    result["score"] = len(result["critical"]) * 100 + len(result["warning"]) * 10 + len(result["info"]) * 1

    return result

def score_all_entries(zidian: Dict[str, Any]) -> List[Tuple[str, int, List[str], List[str], List[str]]]:
    """
    遍历整个字典，对每个词条打分，返回按总分降序排序的列表。
    每个元素: (汉字, 总分, critical列表, warning列表, info列表)
    """
    results = []
    for zi in zidian.keys():
        s = score_sanity(zi, zidian)
        results.append((zi, s["score"], s["critical"], s["warning"], s["info"]))
    # 按总分降序排序
    results.sort(key=lambda x: x[1], reverse=True)
    return results

# 使用示例（假设 zidian 已存在）
sorted_scores = score_all_entries(zid)

nbar = 90
print(sum([int(score > nbar) for zi, score, crit, warn, info in sorted_scores]))
# 打印前20个最严重的问题词条
for zi, score, crit, warn, info in sorted_scores[:30]:
    print(f"{zi} (总分 {score}): Critical={len(crit)}, Warning={len(warn)}, Info={len(info)}")
    if crit:
        print(f"  严重: {crit[:2]}...")  # 只显示前两条

2
哓 (总分 100): Critical=1, Warning=0, Info=0
  严重: ['基本解释第1项 第1个义项：释义为空']...
髃 (总分 100): Critical=1, Warning=0, Info=0
  严重: ['基本解释第1项 第2个义项：释义为空']...
騷 (总分 30): Critical=0, Warning=3, Info=0
驃 (总分 30): Critical=0, Warning=3, Info=0
鰓 (总分 30): Critical=0, Warning=3, Info=0
鱖 (总分 30): Critical=0, Warning=3, Info=0
鱣 (总分 30): Critical=0, Warning=3, Info=0
鵠 (总分 30): Critical=0, Warning=3, Info=0
鸛 (总分 30): Critical=0, Warning=3, Info=0
鼰 (总分 30): Critical=0, Warning=3, Info=0
齗 (总分 30): Critical=0, Warning=3, Info=0
齦 (总分 30): Critical=0, Warning=3, Info=0
偊 (总分 20): Critical=0, Warning=2, Info=0
儽 (总分 20): Critical=0, Warning=2, Info=0
呣 (总分 20): Critical=0, Warning=2, Info=0
圙 (总分 20): Critical=0, Warning=2, Info=0
帩 (总分 20): Critical=0, Warning=2, Info=0
棯 (总分 20): Critical=0, Warning=2, Info=0
檷 (总分 20): Critical=0, Warning=2, Info=0
汦 (总分 20): Critical=0, Warning=2, Info=0
沨 (总分 20): Critical=0, Warning=2, Info=0
炘 (总分 20): Critical=0, Warning=2, Info=0
畟 (总分 20): Critical=0, Warning

In [223]:
sorted_scores[2]

('标',
 30,
 [],
 ['详解第1项 第15个义项：「又如」义项中不应包含英文标记，可能解析错误',
  '详解第1项 第16个义项：「又如」义项中不应包含英文标记，可能解析错误',
  '详解第1项 第17个义项：「又如」义项中不应包含英文标记，可能解析错误'],
 [])

In [227]:
zid['标']['详解'][0]['义'][14]

['又如:标的(准则,法则;标志,记号;靶子) 目标', '[target]', '大会射,设标的。--韩愈《国子助教薛君墓志铭》']

In [173]:
xinhua['騷']

'基本解释\nBasic explanation\n騷\nsāo\n详细解析\nDetailed explanation\n騷\nsāo\n中文輸入法 Input Methods\n仓颉 Cāngjié\nSFEII\n郑码 Zhèngmǎ\nCUSI\n四角号码 Four Corner\n77336\nUnicode\nU+9A37'

In [219]:
print(xinhua['苯'])

基本解释
Basic explanation
苯　　běn  ㄅㄣˇ
◎ 一种有机化合物，无色液体，有特殊的气味，可从煤焦油，石油中提取，是多种化学工业的原料和溶剂。
详细解析
Detailed explanation
◎ 苯 běn
〈形〉
词性变化
◎ 苯 běn
〈名〉
无色、挥发、可燃的毒性液体芳烃C 6 H 6
[benzene],遇火燃烧。商品是从煤的炼焦(如从焦炉气的轻油中)或从某些石油馏份通过催化脱氢获得,主要用于有机合成
中文輸入法 Input Methods
仓颉 Cāngjié
TDM
郑码 Zhèngmǎ
EFA
四角号码 Four Corner
44234
Unicode
U+82EF
